# 05. Regular expressions

**Tier:** Core. **Why you'd use this:** Log lines mix fixed words with ids and numbers. You need every id, or a cleaned line with ids blanked out, without writing a fragile chain of `split` calls.

**After this lesson you can:** use `findall` and `sub` on a pattern; explain why a pattern is often a raw string; spot a greedy match that swallows too much.
About 25 minutes. **Uses:** [Format specs](04-specs-lesson.ipynb)


## Quick review

1. What does `f"{3.14159:.2f}"` print?
2. Does a format width shorten a longer string?

<details><summary>Answers</summary>

`3.14`. No — width pads; it does not truncate.
</details>


## The idea

A regular expression is a pattern for text, from the `re` module. `findall` returns every match. `sub` replaces matches. `\d+` is one or more digits. Write the pattern as a raw string, `r"..."`, so backslashes reach the engine.

<details><summary>Explain more</summary>

A group in parentheses is the part `findall` returns. Python applies string escapes before `re` reads the pattern.
</details>


## See it work

Pull three-digit area codes out of a sentence. The outer parentheses are real characters. The inner pair is a group, which is what `findall` returns.


In [1]:
import re


def area_codes(text):
    # 1. Three digits with a parenthesis on each side.
    # 2. The group is what findall keeps.
    return re.findall(r"\((\d{3})\)", text)


print(area_codes("call (415) or (628) today"))

['415', '628']


## Check yourself

Answer from the example before you run.


1. What does the engine see in `"\bcat\b"` that it does not see in the raw pattern?
2. What does `findall` return from `area_codes`, which has a group?

<details><summary>Answers</summary>

The normal string is backspaces around `cat`. The raw string keeps the backslashes (`\b` = word boundary). With a group, `findall` returns the groups.
</details>


In [2]:
print(re.findall(r"\d+", "room 12 and 3"))
print(re.findall(r"\bcat\b", "the cat sat"))
print(repr("\bcat\b"))
print(re.findall("\bcat\b", "the cat sat"))
print(re.sub(r"red", "blue", "a red door"))

['12', '3']
['cat']
'\x08cat\x08'
[]
a blue door


<details><summary>Check your answers</summary>

- `\d+` keeps `12` and `3`.
- The raw pattern finds `cat` as a whole word. `repr` shows that a normal `"\b"` is one backspace, so the same-looking pattern matches nothing.
- `sub` replaces `red` and leaves the rest.
</details>


## Why it works this way

A pattern is text the `re` engine reads after Python has already applied string escapes. In a normal literal, `\b` is one backspace, so the match is empty — a silent failure. A raw string keeps the backslash: `r"\b"` is what the engine reads as a word edge. [Numbers and strings](../../internals/05-numbers/01-numbers-strings-lesson.ipynb) unpacks the literal.

`*` is greedy; `?` after it takes as few as possible. `(a+)+` on a failing run of `a`s tries every split between the inner and outer repeat — catastrophic backtracking. The timed cell fails at 16 `a`s and again at 20. Do not nest repeats like that. `re.compile` pays the parse once when the same pattern runs often.


In [3]:
import timeit

pattern = re.compile(r"(a+)+$")
print(
    "16",
    timeit.timeit(
        "pattern.match(text)",
        setup="text = 'a' * 16 + '!'",
        number=1,
        globals=globals(),
    ),
)
print(
    "20",
    timeit.timeit(
        "pattern.match(text)",
        setup="text = 'a' * 20 + '!'",
        number=1,
        globals=globals(),
    ),
)

16 0.0029542080010287464
20 0.043082417018013075


## When to use it

Use a regular expression for optional pieces, repeats, or a group to keep. Use `split` for a fixed separator. Do not parse HTML this way. A greedy `.*` crosses boundaries you meant to keep. Compile a pattern you reuse so the engine builds it once.


## Common mistake

A greedy `.*` between quotes runs from the first quote to the last one that still lets the pattern finish. One match swallows both phrases.


In [4]:
import re

text = 'say "one" and "two"'
print(re.findall(r'".*"', text))

['"one" and "two"']


`?` after `*` stops at the first closing quote. `findall` then resumes, and the second phrase is its own match.


In [5]:
import re

text = 'say "one" and "two"'
print(re.findall(r'".*?"', text))

['"one"', '"two"']


## Try it

Change `area_codes` so `( 415)` matches as well as `(415)`. What do you add, and why does `\s*` belong beside the opening parenthesis rather than only beside the digits?


## Recap

- `findall` returns every match, or the groups if you wrote any.
- A raw string lets backslashes through. `"\b"` is a backspace.
- Nested repeats can explode on a failure.

Next: [practice](05-regex-practice.ipynb).
